### John (Jack) Leniart
#### CSC555 - Final Project

-----------------------

#### Loading Data and Initial Preprocessing

In [1]:
import pyspark # only run after findspark.init()
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Classification").getOrCreate()

cores = spark._jsc.sc().getExecutorMemoryStatus().keySet().size()
print("You are working with", cores, "core(s)")
spark

VBox()

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
0,application_1731984268094_0001,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

You are working with 4 core(s)

In [2]:
sc.install_pypi_package("numpy")
sc.install_pypi_package("pandas")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…


  Attempting uninstall: python-dateutil
    Found existing installation: python-dateutil 2.8.1
    Not uninstalling python-dateutil at /usr/lib/python3.9/site-packages, outside environment /mnt/yarn/usercache/livy/appcache/application_1731984268094_0001/container_1731984268094_0001_01_000001/tmp/spark-c440eca4-f2d2-4d5b-9315-080c67d1bf63
    Can't uninstall 'python-dateutil'. No files were found to uninstall.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
awscli 2.15.30 requires python-dateutil<=2.8.2,>=2.1, but you have python-dateutil 2.9.0.post0 which is incompatible.

In [3]:
# Read full dataset from S3
fraud_df = spark.read.csv('s3://leniart-final-bucket/Fraud.csv',inferSchema=True,header=True)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

*About Dataset:* This dataset I am using for this project is the Fraudulent Transactions dataset from Kaggle. The dataset contains simulated financial transactions. Each transaction is represented by a row in the dataset. Some of the transactions are labeled as legitimate and some are labeled as fraudulent. The dataset contains 6,362,620 rows and 11 columns.

*Source:* https://www.kaggle.com/datasets/vardhansiramdasu/fraudulent-transactions-prediction/data

In [4]:
# Preview data
fraud_df.limit(10).toPandas()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

   step      type    amount  ... newbalanceDest  isFraud  isFlaggedFraud
0     1   PAYMENT   9839.64  ...           0.00        0               0
1     1   PAYMENT   1864.28  ...           0.00        0               0
2     1  TRANSFER    181.00  ...           0.00        1               0
3     1  CASH_OUT    181.00  ...           0.00        1               0
4     1   PAYMENT  11668.14  ...           0.00        0               0
5     1   PAYMENT   7817.71  ...           0.00        0               0
6     1   PAYMENT   7107.77  ...           0.00        0               0
7     1   PAYMENT   7861.64  ...           0.00        0               0
8     1   PAYMENT   4024.36  ...           0.00        0               0
9     1     DEBIT   5337.77  ...       40348.79        0               0

[10 rows x 11 columns]

In [5]:
# Schema
fraud_df.printSchema()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- step: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: integer (nullable = true)
 |-- isFlaggedFraud: integer (nullable = true)

In [6]:
# Drop missing data
fraud_df = fraud_df.na.drop()
fraud_df.count()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

6362620

In [7]:
# Check the total counts for eaach class in our target variable
fraud_df.groupBy("isFraud").count().show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-------+-------+
|isFraud|  count|
+-------+-------+
|      1|   8213|
|      0|6354407|
+-------+-------+

In [7]:
# Check the total counts for each value of the type variable
# We will be creating dummy variables for these
fraud_df.groupBy("type").count().show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------+-------+
|    type|  count|
+--------+-------+
|CASH_OUT|2237500|
| CASH_IN|1399284|
| PAYMENT|2151495|
|   DEBIT|  41432|
|TRANSFER| 532909|
+--------+-------+

In [10]:
# Add a new column with row IDs so that we can group by id when pivoting to create dummy variables
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, LongType

row_with_index = Row("step","type","amount","nameOrig","oldbalanceOrg","newbalanceOrig","nameDest","oldbalanceDest","newbalanceDest","isFraud","isFlaggedFraud","id")

new_schema = StructType(fraud_df.schema.fields[:] + [StructField("id", LongType(), False)])
zipped_rdd = fraud_df.rdd.zipWithIndex()
indexed_fraud_df = (zipped_rdd.map(lambda ri: row_with_index(*list(ri[0]) + [ri[1]])).toDF(new_schema))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [11]:
indexed_fraud_df.show(10)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----+--------+--------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+---+
|step|    type|  amount|   nameOrig|oldbalanceOrg|newbalanceOrig|   nameDest|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud| id|
+----+--------+--------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+---+
|   1| PAYMENT| 9839.64|C1231006815|     170136.0|     160296.36|M1979787155|           0.0|           0.0|      0|             0|  0|
|   1| PAYMENT| 1864.28|C1666544295|      21249.0|      19384.72|M2044282225|           0.0|           0.0|      0|             0|  1|
|   1|TRANSFER|   181.0|C1305486145|        181.0|           0.0| C553264065|           0.0|           0.0|      1|             0|  2|
|   1|CASH_OUT|   181.0| C840083671|        181.0|           0.0|  C38997010|       21182.0|           0.0|      1|             0|  3|
|   1| PAYMENT|11668.14|C2048537720|      41554.0|     

In [12]:
# Create dummy variables for the different values for type
from pyspark.sql import functions as F
fraud_type_dummy_df = indexed_fraud_df.groupBy("id").pivot("type").agg(F.lit(1))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [13]:
# Replace all NULLs in the dummy variables with 0
fraud_type_dummy_df = fraud_type_dummy_df.na.fill(0)
fraud_type_dummy_df.show(10)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-------+-------+--------+-----+-------+--------+
|     id|CASH_IN|CASH_OUT|DEBIT|PAYMENT|TRANSFER|
+-------+-------+--------+-----+-------+--------+
| 638715|      0|       1|    0|      0|       0|
|1258679|      0|       1|    0|      0|       0|
|1194255|      0|       0|    0|      1|       0|
| 176926|      1|       0|    0|      0|       0|
|1053251|      1|       0|    0|      0|       0|
|1389824|      0|       1|    0|      0|       0|
|1699850|      1|       0|    0|      0|       0|
|  85051|      0|       0|    0|      0|       1|
| 601963|      0|       1|    0|      0|       0|
| 922220|      0|       1|    0|      0|       0|
+-------+-------+--------+-----+-------+--------+
only showing top 10 rows

In [16]:
# Join original dataframe and dummy variable dataframe
fraud_full_df = indexed_fraud_df.join(fraud_type_dummy_df, ["id"], how='left')
fraud_full_df.show(10)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-------+----+--------+---------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+-------+--------+-----+-------+--------+
|     id|step|    type|   amount|   nameOrig|oldbalanceOrg|newbalanceOrig|   nameDest|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|CASH_IN|CASH_OUT|DEBIT|PAYMENT|TRANSFER|
+-------+----+--------+---------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+-------+--------+-----+-------+--------+
|1738332| 161| PAYMENT| 18672.42|C2050705555|          0.0|           0.0| M863681574|           0.0|           0.0|      0|             0|      0|       0|    0|      1|       0|
|3465886| 257| PAYMENT| 13091.97| C493522515|      18587.0|       5495.03|M1348000224|           0.0|           0.0|      0|             0|      0|       0|    0|      1|       0|
|5193250| 369|CASH_OUT|213611.47|C1011201006|      10102.0|           0.0| C527743644|           0.0

In [17]:
# Check dimensions to verify join completed as expected
print("Total rows in original dataframe: ", indexed_fraud_df.count())
print("Total rows in dummy dataframe: ", fraud_type_dummy_df.count())
print("Total rows in full dataframe: ", fraud_full_df.count())
print("Total columns in full dataframe: ", len(fraud_full_df.columns))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Total rows in original dataframe:  6362620
Total rows in dummy dataframe:  6362620
Total rows in full dataframe:  6362620
Total columns in full dataframe:  17

----------------------------

#### Athena

In [19]:
sc.install_pypi_package("boto3")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [20]:
import boto3
import pandas as pd
import numpy as np

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [21]:
#Set Variables
# AWS configuration (no need to specify credentials)
AWS_REGION = 'us-east-1'  # Replace with your AWS region
S3_BUCKET_NAME = 'leniart-final-bucket'
S3_KEY_DATA = 'fraud_data/'
S3_KEY_ATHENA = 'athena/'
filename = 'Fraud.csv'

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [23]:
# Initialize S3 client (no need to specify credentials)
s3 = boto3.client('s3', region_name=AWS_REGION)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [25]:
# Write the updated full dataset with dummy variables to a CSV file and upload it to s3 bucket
path ="s3://leniart-final-bucket/"
fraud_full_df.coalesce(1).write.mode("overwrite").csv(path+S3_KEY_DATA)
fraud_full_df.coalesce(1).write.mode("overwrite").csv(path+S3_KEY_ATHENA)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [26]:
# Initialize Athena client (no need to specify credentials)
athena_client = boto3.client('athena', region_name=AWS_REGION)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [27]:
#Need to update with access/session info
athena_client = boto3.client('athena',
                             aws_access_key_id='',
                             aws_secret_access_key='',
                             aws_session_token='', #your session token
                             region_name = AWS_REGION)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [28]:
# Athena configuration
DATABASE_NAME = 'default'
TABLE_NAME = 'fraud_table_leniart'

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [30]:
# Define the table creation query
create_table_query = f"""
CREATE EXTERNAL TABLE IF NOT EXISTS `{DATABASE_NAME}`.`{TABLE_NAME}` (
    id INT,
    step INT,
    type STRING,
    amount FLOAT,
    nameOrig STRING,
    oldbalanceOrg FLOAT,
    newbalanceOrig FLOAT,
    nameDest STRING,
    oldbalanceDest FLOAT,
    newbalanceDest FLOAT,
    isFraud INT,
    isFlaggedFraud INT,
    CASH_IN INT,
    CASH_OUT INT,
    DEBIT INT,
    PAYMENT INT,
    TRANSFER INT
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
LOCATION 's3://{S3_BUCKET_NAME}/athena'
"""

print(create_table_query.strip())

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

CREATE EXTERNAL TABLE IF NOT EXISTS `default`.`fraud_table_leniart` (
    id INT,
    step INT,
    type STRING,
    amount FLOAT,
    nameOrig STRING,
    oldbalanceOrg FLOAT,
    newbalanceOrig FLOAT,
    nameDest STRING,
    oldbalanceDest FLOAT,
    newbalanceDest FLOAT,
    isFraud INT,
    isFlaggedFraud INT,
    CASH_IN INT,
    CASH_OUT INT,
    DEBIT INT,
    PAYMENT INT,
    TRANSFER INT
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
LOCATION 's3://leniart-final-bucket/athena'

In [31]:
# Execute query to create table
response = athena_client.start_query_execution(
    QueryString=create_table_query.strip(),
    QueryExecutionContext={'Database': DATABASE_NAME},
    ResultConfiguration={'OutputLocation': f's3://{S3_BUCKET_NAME}/query_results/'}
)

print("Athena table creation query submitted.")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Athena table creation query submitted.

In [32]:
# Define Python functions to run Athena query, output results to S3, and then read results from S3 into dataframe
import time

# Function to run Athena query
def run_athena_query(query, database, output_location):
    response = athena_client.start_query_execution(
        QueryString=query,
        QueryExecutionContext={
            'Database': database
        },
        ResultConfiguration={
            'OutputLocation': output_location
        }
    )
    return response['QueryExecutionId']

# Function to check query status
def check_query_status(query_execution_id):
    while True:
        response = athena_client.get_query_execution(QueryExecutionId=query_execution_id)
        status = response['QueryExecution']['Status']['State']
        if status == 'SUCCEEDED':
            print("Query succeeded!")
            break
        elif status in ['FAILED', 'CANCELLED']:
            print(f"Query {status.lower()}.")
            raise Exception(f"Query failed or was cancelled: {response}")
        time.sleep(2)

# Function to get query results and convert to pandas DataFrame
def get_query_results_as_pd_dataframe(query_execution_id):
    # Fetching the query results
    results_paginator = athena_client.get_paginator('get_query_results')
    results_iter = results_paginator.paginate(QueryExecutionId=query_execution_id)

    # Initialize a list to store rows and columns for DataFrame
    columns = []
    rows = []

    # Process the result pages
    for results_page in results_iter:
        # Get column info from the first page
        if not columns:
            columns = [col['Label'] for col in results_page['ResultSet']['ResultSetMetadata']['ColumnInfo']]

        # Skip the first row of the first page (column headers)
        for row in results_page['ResultSet']['Rows'][1:]:
            rows.append([col.get('VarCharValue', None) for col in row['Data']])

    # Create DataFrame from the results
    df = pd.DataFrame(rows, columns=columns)
    return df

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [33]:
# Run a test query to pull data from the new table in Athena
test_query = f'SELECT * FROM "{DATABASE_NAME}"."{TABLE_NAME}" limit 10;'
database = DATABASE_NAME
output_location = f's3://{S3_BUCKET_NAME}/query_results/'

# Run the query
test_query_execution_id = run_athena_query(test_query, database, output_location)

# Check the status of the query
check_query_status(test_query_execution_id)

# Fetch the results as a pandas DataFrame - this will not work with large dataset if limit is not set
df = get_query_results_as_pd_dataframe(test_query_execution_id)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Query succeeded!

In [34]:
df

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

        id step      type     amount  ... cash_out debit payment transfer
0  3556594  260   CASH_IN  187669.62  ...        0     0       0        0
1  3556620  260  CASH_OUT  145095.62  ...        1     0       0        0
2  3556631  260  TRANSFER  117056.33  ...        0     0       0        1
3  3556673  260  CASH_OUT   35489.47  ...        1     0       0        0
4  3556675  260  CASH_OUT  278527.53  ...        1     0       0        0
5  3556682  260   PAYMENT    4409.12  ...        0     0       1        0
6  3556692  260   PAYMENT    7065.13  ...        0     0       1        0
7  3556712  260   PAYMENT   12802.75  ...        0     0       1        0
8  3556740  260   PAYMENT    9884.57  ...        0     0       1        0
9  3556757  260   PAYMENT    9592.41  ...        0     0       1        0

[10 rows x 17 columns]

In [35]:
# Some additional data preprocessing using an Athena query
# For my model, I do not want to use the following variables: nameOrig, nameDest, isFlaggedFraud
# In addition to that, I want to add two new columns to the dataset
# The two new variables will store the changes in balance for the original and destination accounts

# Query details
bal_diff_query = f"""
SELECT amount, oldbalanceOrg, newbalanceOrig,(newbalanceOrig - oldbalanceOrg) AS BalChangeOrig,
oldbalanceDest, newbalanceDest,(newbalanceDest - oldbalanceDest) AS BalChangeDest, isFraud,
CASH_IN, CASH_OUT, DEBIT, PAYMENT, TRANSFER
FROM "{DATABASE_NAME}"."{TABLE_NAME}";
"""
database = DATABASE_NAME
output_location = f's3://{S3_BUCKET_NAME}/query_results/'

# Run the query
query_execution_id = run_athena_query(bal_diff_query, database, output_location)

# Check the status of the query
check_query_status(query_execution_id)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Query succeeded!

In [36]:
#Pull query results from S3 bucket using object key
results_path = "s3://leniart-final-bucket/query_results/21b4bc3e-8885-43ed-ba30-04e79af0f307.csv"

fraud_df = spark.read.csv(results_path,inferSchema=True,header=True)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [37]:
fraud_df.show(10)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---------+-------------+--------------+-------------+--------------+--------------+-------------+-------+-------+--------+-----+-------+--------+
|   amount|oldbalanceOrg|newbalanceOrig|BalChangeOrig|oldbalanceDest|newbalanceDest|BalChangeDest|isFraud|CASH_IN|CASH_OUT|DEBIT|PAYMENT|TRANSFER|
+---------+-------------+--------------+-------------+--------------+--------------+-------------+-------+-------+--------+-----+-------+--------+
|  6061.13|        443.0|           0.0|       -443.0|           0.0|           0.0|          0.0|      0|      0|       0|    0|      1|       0|
|  6440.78|       2192.0|           0.0|      -2192.0|           0.0|           0.0|          0.0|      0|      0|       0|    0|      1|       0|
| 12986.61|      23350.0|      10363.39|    -12986.61|           0.0|           0.0|          0.0|      0|      0|       0|    0|      1|       0|
|1724887.0|          0.0|           0.0|          0.0|     3470595.0|   1.9169204E7|  1.5698609E7|      0|      0|    

In [38]:
# Take a closer look at the distribution of the numeric data using the summary function
numeric_cols = ['amount','oldbalanceOrg','newbalanceOrig','BalChangeOrig','oldbalanceDest','newbalanceDest','BalChangeDest']
fraud_df.select(numeric_cols).summary("count", "min", "25%", "50%", "75%", "max").show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-------+----------+-------------+--------------+-------------+--------------+--------------+-------------+
|summary|    amount|oldbalanceOrg|newbalanceOrig|BalChangeOrig|oldbalanceDest|newbalanceDest|BalChangeDest|
+-------+----------+-------------+--------------+-------------+--------------+--------------+-------------+
|  count|   6362620|      6362620|       6362620|      6362620|       6362620|       6362620|      6362620|
|    min|       0.0|          0.0|           0.0| -1.0000002E7|           0.0|           0.0| -1.3060826E7|
|    25%|  13385.86|          0.0|           0.0|     -10151.0|           0.0|           0.0|          0.0|
|    50%|  74853.59|     14206.82|           0.0|          0.0|     132661.19|     214685.81|          0.0|
|    75%| 208704.77|     107307.0|      144210.4|          0.0|      942748.7|     1111766.5|    149079.97|
|    max|9.244552E7|   5.958504E7|    4.958504E7|    1915268.0|  3.56015904E8|  3.56179264E8|  1.0568784E8|
+-------+----------+--------

In [39]:
# And also the describe function
fraud_df.describe(numeric_cols).show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+
|summary|            amount|     oldbalanceOrg|   newbalanceOrig|     BalChangeOrig|    oldbalanceDest|    newbalanceDest|     BalChangeDest|
+-------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+
|  count|           6362620|           6362620|          6362620|           6362620|           6362620|           6362620|           6362620|
|   mean|179861.90355834848|  833883.104048853| 855113.668553137|21230.564503113146|1100701.6665187487|1224996.3981876222|124294.73167288191|
| stddev| 603858.2316747018|2888242.6729981783|2924048.502909438| 146643.2864296386|3399180.1117589492|  3674128.94101535| 812939.0805457995|
|    min|               0.0|               0.0|              0.0|      -1.0000002E7|               0.0|               0.0|      -1.3060826E7|
|    m

In [40]:
# Import Libraries

# For data prep
from pyspark.sql.types import * 
from pyspark.sql.functions import *
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.feature import StringIndexer
from pyspark.ml.feature import MinMaxScaler

# For training and evaluation
from pyspark.ml.evaluation import *
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.classification import *

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [41]:
# Drop missing data - I noticed the Athena query results were adding an additional row with null values
fraud_df = fraud_df.na.drop()
fraud_df.count()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

6362620

---------------------

#### PySpark - Format Data for Classification Model - Vectorize

In [48]:
# Declare values you will need
input_columns = fraud_df.columns # Collect the column names as a list
independent_columns = input_columns[:-1] # keep only independent variables (all columns except isFraud) 
dependent_var = 'isFraud'

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [49]:
# change label (class variable) to string type to prep for reindexing
# Pyspark is expecting a zero indexed integer for the label column. 
# Just in case our data is not in that format... we will treat it by using the StringIndexer built in method

renamed = fraud_df.withColumn("label_str", fraud_df[dependent_var].cast(StringType())) #Rename and change to string type
indexer = StringIndexer(inputCol="label_str", outputCol="label") #Pyspark is expecting this naming convention 
fraud_indexed_df = indexer.fit(renamed).transform(renamed)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [50]:
# Convert all string type data in the input column list to numeric
# Otherwise the Algorithm will not be able to process it

# Also we will use these lists later on
numeric_inputs = []
string_inputs = []
for column in input_columns:
    # First identify the string vars in your input column list
    if str(fraud_indexed_df.schema[column].dataType) == 'StringType()':
        # Set up your String Indexer function
        indexer = StringIndexer(inputCol=column, outputCol=column+"_num") 
        # Then call on the indexer you created here
        fraud_indexed_df = indexer.fit(fraud_indexed_df).transform(fraud_indexed_df)
        # Rename the column to a new name so you can disinguish it from the original
        new_col_name = column+"_num"
        # Add the new column name to the string inputs list
        string_inputs.append(new_col_name)
    else:
        # If no change was needed, take no action 
        # And add the numeric var to the num list
        numeric_inputs.append(column)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [55]:
numeric_inputs

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

['amount', 'oldbalanceOrg', 'newbalanceOrig', 'BalChangeOrig', 'oldbalanceDest', 'newbalanceDest', 'BalChangeDest', 'isFraud', 'CASH_IN', 'CASH_OUT', 'DEBIT', 'PAYMENT', 'TRANSFER']

In [52]:
string_inputs

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

[]

In [56]:
#Remove the dummy variables from this list because, although they hold integers, they are categorical
numeric_inputs.remove('CASH_IN')
numeric_inputs.remove('CASH_OUT')
numeric_inputs.remove('DEBIT')
numeric_inputs.remove('PAYMENT')
numeric_inputs.remove('TRANSFER')
numeric_inputs

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

['amount', 'oldbalanceOrg', 'newbalanceOrig', 'BalChangeOrig', 'oldbalanceDest', 'newbalanceDest', 'BalChangeDest', 'isFraud']

In [57]:
# Maybe skip
# Treat for skewness
# Flooring and capping
# If right skew, take the log +1
# If left skew, do exp transformation

# create empty dictionary d
d = {}

# Create a dictionary of quantiles from your numeric cols - top and bottom 1%
for col in numeric_inputs: 
    d[col] = fraud_indexed_df.approxQuantile(col,[0.01,0.99],0.25) #if you want to make it go faster increase the last number

#Now check for skewness for all numeric cols
for col in numeric_inputs:
    skew = fraud_indexed_df.agg(skewness(fraud_indexed_df[col])).collect() #check for skewness
    skew = skew[0][0]
    # If skewness is found,
    # This function will make the appropriate corrections
    if skew > 1: # If right skew, floor, cap and log(x+1)
        fraud_indexed_df = fraud_indexed_df.withColumn(col, \
        log(when(fraud_df[col] < d[col][0],d[col][0])\
        .when(fraud_indexed_df[col] > d[col][1], d[col][1])\
        .otherwise(fraud_indexed_df[col] ) +1).alias(col))
        print(col+" has been treated for positive (right) skewness. (skew =)",skew,")")
    elif skew < -1: # If left skew floor, cap and exp(x)
        fraud_indexed_df = fraud_indexed_df.withColumn(col, \
        exp(when(fraud_df[col] < d[col][0],d[col][0])\
        .when(fraud_indexed_df[col] > d[col][1], d[col][1])\
        .otherwise(fraud_indexed_df[col] )).alias(col))
        print(col+" has been treated for negative (left) skewness. (skew =",skew,")")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

amount has been treated for positive (right) skewness. (skew =) 30.993942214247106 )
oldbalanceOrg has been treated for positive (right) skewness. (skew =) 5.249135183596566 )
newbalanceOrig has been treated for positive (right) skewness. (skew =) 5.176882781152682 )
BalChangeOrig has been treated for negative (left) skewness. (skew = -24.63051458863992 )
oldbalanceDest has been treated for positive (right) skewness. (skew =) 19.921753203583236 )
newbalanceDest has been treated for positive (right) skewness. (skew =) 19.35229746998054 )
BalChangeDest has been treated for positive (right) skewness. (skew =) 32.916332825240445 )
isFraud has been treated for positive (right) skewness. (skew =) 27.77953160398267 )

In [58]:
# Vectorize our df because the function that we use to make that correction requires a vector
# Create final features list
features_list = ['amount','oldbalanceOrg','newbalanceOrig','BalChangeOrig','oldbalanceDest','newbalanceDest','BalChangeDest','CASH_IN','CASH_OUT','DEBIT','PAYMENT','TRANSFER']

# Create vector assembler object
assembler = VectorAssembler(handleInvalid='skip',inputCols=features_list,outputCol='features')
#could also try handleInvalid='keep', but we still have over 5million transactions after the skips

# And call on the vector assembler to transform your dataframe
output = assembler.transform(fraud_indexed_df).select('features','label')

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [59]:
output.count() #to see how many rows were skipped by VectorAssembler

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

5123763

In [60]:
# Maybe Skip
# Create the min max scaler object and use it to correct for negative values
# Use a high range like 1,000 because I only see one decimal place in the final_data.show() call

scaler = MinMaxScaler(inputCol="features", outputCol="scaledFeatures",min=0,max=1000)
print("Features scaled to range: [%f, %f]" % (scaler.getMin(), scaler.getMax()))

# Compute summary statistics and generate MinMaxScalerModel
scalerModel = scaler.fit(output)

# Rescale each feature to range [min, max].
scaled_data = scalerModel.transform(output)
final_data = scaled_data.select('label','scaledFeatures')

# Rename to default value
final_data = final_data.withColumnRenamed("scaledFeatures","features")
final_data.show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Features scaled to range: [0.000000, 1000.000000]
+-----+--------------------+
|label|            features|
+-----+--------------------+
|  0.0|(12,[0,1,3,6,10],...|
|  0.0|(12,[0,1,3,6,10],...|
|  0.0|(12,[0,1,2,3,6,10...|
|  0.0|(12,[0,3,4,5,6,11...|
|  0.0|(12,[0,1,2,3,6,10...|
|  0.0|(12,[0,1,2,3,6,10...|
|  0.0|(12,[0,3,4,5,6,8]...|
|  0.0|(12,[0,1,2,3,6,10...|
|  0.0|[652.575039080977...|
|  0.0|(12,[0,1,2,3,6,10...|
|  0.0|(12,[0,1,3,6,10],...|
|  0.0|[671.330799718190...|
|  0.0|(12,[0,1,2,3,6,10...|
|  0.0|(12,[0,1,2,3,6,10...|
|  0.0|(12,[0,1,2,3,6,10...|
|  0.0|(12,[0,3,6,10],[5...|
|  0.0|(12,[0,1,2,3,6,10...|
|  0.0|[652.882369288009...|
|  0.0|(12,[0,1,2,3,6,10...|
|  0.0|(12,[0,3,4,5,6,8]...|
+-----+--------------------+
only showing top 20 rows

--------------------

#### Split Into Train and Test

In [61]:
train_df, test_df = final_data.randomSplit([0.7,0.3])

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [62]:
# Set up our evaluation objects
from pyspark.ml.evaluation import BinaryClassificationEvaluator
Bin_evaluator = BinaryClassificationEvaluator(rawPredictionCol='prediction') #labelCol='label'
# Bin_evaluator = BinaryClassificationEvaluator() #labelCol='label'

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

---------------

#### Classification Model 1 - Linear Support Vector Machine (SVM)

In [63]:
# Add parameters of your choice here:
SVM_classifier = LinearSVC()
SVM_paramGrid = (ParamGridBuilder() \
             .addGrid(SVM_classifier.maxIter, [10, 15]) \
             .addGrid(SVM_classifier.regParam, [0.1, 0.01]) \
             .build())

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [64]:
#Cross Validator requires all of the following parameters:
SVM_crossval = CrossValidator(estimator=SVM_classifier,
                          estimatorParamMaps=SVM_paramGrid,
                          evaluator=MulticlassClassificationEvaluator(),
                          numFolds=5) # 3 + is best practice

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [65]:
# Fit Model: Run cross-validation, and choose the best set of parameters.
fit_SVM_Model = SVM_crossval.fit(train_df)

Best_SVM_Model = fit_SVM_Model.bestModel

print("Intercept: \n" + str(Best_SVM_Model.intercept))
print(" ")
print('\033[1m' + " Coefficients"+ '\033[0m')
print("Compare coefficients relative to eachother")
print("Coefficients: \n" + str(Best_SVM_Model.coefficients))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Exception in thread cell_monitor-65:
Traceback (most recent call last):
  File "/mnt/notebook-env/lib/python3.9/threading.py", line 980, in _bootstrap_inner
    self.run()
  File "/mnt/notebook-env/lib/python3.9/threading.py", line 917, in run
    self._target(*self._args, **self._kwargs)
  File "/mnt/notebook-env/lib/python3.9/site-packages/awseditorssparkmonitoringwidget/cellmonitor.py", line 154, in cell_monitor
    job_group_filtered_jobs = [job for job in jobs_data if job['jobGroup'] == str(statement_id)]
  File "/mnt/notebook-env/lib/python3.9/site-packages/awseditorssparkmonitoringwidget/cellmonitor.py", line 154, in <listcomp>
    job_group_filtered_jobs = [job for job in jobs_data if job['jobGroup'] == str(statement_id)]
KeyError: 'jobGroup'


Intercept: 
-1.0998019278528002
 
 Coefficients
Compare coefficients relative to eachother
Coefficients: 
[6.267515456169145e-05,3.2784654359030296e-05,-1.5177594506238201e-05,0.0,-6.7807865667876535e-06,-4.514306034524153e-06,-4.27616381092532e-06,-7.704912763255276e-06,3.370415296449665e-06,-7.454987232988239e-06,-1.2748123210360831e-05,2.3398994197630443e-05]

In [66]:
# Automatically gets the best model
SVM_predictions = fit_SVM_Model.transform(test_df)
SVM_accuracy = (Bin_evaluator.evaluate(SVM_predictions))*100
print("Accuracy: ", SVM_accuracy)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Exception in thread cell_monitor-66:
Traceback (most recent call last):
  File "/mnt/notebook-env/lib/python3.9/threading.py", line 980, in _bootstrap_inner
    self.run()
  File "/mnt/notebook-env/lib/python3.9/threading.py", line 917, in run
    self._target(*self._args, **self._kwargs)
  File "/mnt/notebook-env/lib/python3.9/site-packages/awseditorssparkmonitoringwidget/cellmonitor.py", line 154, in cell_monitor
    job_group_filtered_jobs = [job for job in jobs_data if job['jobGroup'] == str(statement_id)]
  File "/mnt/notebook-env/lib/python3.9/site-packages/awseditorssparkmonitoringwidget/cellmonitor.py", line 154, in <listcomp>
    job_group_filtered_jobs = [job for job in jobs_data if job['jobGroup'] == str(statement_id)]
KeyError: 'jobGroup'


Accuracy:  50.0

In [84]:
#Generate Confusion Matrix to review results of predictions
#Cast to float type, and order by prediction, else it won't work
SVM_preds_and_labels = SVM_predictions.select(['prediction','label']).withColumn('label', F.col('label').cast(FloatType())).orderBy('prediction')

#select only prediction and label columns
SVM_preds_and_labels = SVM_preds_and_labels.select(['prediction','label'])

from pyspark.mllib.evaluation import MulticlassMetrics
SVM_metrics = MulticlassMetrics(SVM_preds_and_labels.rdd.map(tuple))

print(SVM_metrics.confusionMatrix().toArray())

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Exception in thread cell_monitor-84:
Traceback (most recent call last):
  File "/mnt/notebook-env/lib/python3.9/threading.py", line 980, in _bootstrap_inner
    self.run()
  File "/mnt/notebook-env/lib/python3.9/threading.py", line 917, in run
    self._target(*self._args, **self._kwargs)
  File "/mnt/notebook-env/lib/python3.9/site-packages/awseditorssparkmonitoringwidget/cellmonitor.py", line 154, in cell_monitor
    job_group_filtered_jobs = [job for job in jobs_data if job['jobGroup'] == str(statement_id)]
  File "/mnt/notebook-env/lib/python3.9/site-packages/awseditorssparkmonitoringwidget/cellmonitor.py", line 154, in <listcomp>
    job_group_filtered_jobs = [job for job in jobs_data if job['jobGroup'] == str(statement_id)]
KeyError: 'jobGroup'


[[1534533.       0.]
 [   2343.       0.]]

In [85]:
print("Summary of evaluation metrics for Linear SVM model:")
print("Accuracy: ", SVM_metrics.accuracy)
print("Precision: ", SVM_metrics.weightedPrecision)
print("Recall: ", SVM_metrics.weightedRecall)
print("F1 Score: ", SVM_metrics.weightedFMeasure())

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Summary of evaluation metrics for Linear SVM model:
Accuracy:  0.9984754788284806
Precision:  0.9969532818217637
Recall:  0.9984754788284806
F1 Score:  0.9977137997271643

Exception in thread cell_monitor-85:
Traceback (most recent call last):
  File "/mnt/notebook-env/lib/python3.9/threading.py", line 980, in _bootstrap_inner
    self.run()
  File "/mnt/notebook-env/lib/python3.9/threading.py", line 917, in run
    self._target(*self._args, **self._kwargs)
  File "/mnt/notebook-env/lib/python3.9/site-packages/awseditorssparkmonitoringwidget/cellmonitor.py", line 154, in cell_monitor
    job_group_filtered_jobs = [job for job in jobs_data if job['jobGroup'] == str(statement_id)]
  File "/mnt/notebook-env/lib/python3.9/site-packages/awseditorssparkmonitoringwidget/cellmonitor.py", line 154, in <listcomp>
    job_group_filtered_jobs = [job for job in jobs_data if job['jobGroup'] == str(statement_id)]
KeyError: 'jobGroup'


The Linear SVM model had a weighted accuracy of 99.84%, but that was largely due to the fact that the majority of rows in the test data had isFraud = 0. You can see in the confusion matrix that the SVM model predict all rows to have isFraud = 0. The model completely overfit to the non-fraud rows.

This performance is a bit underwhelming. Let's proceed with developing some different ML classification models.

------------------------

#### Classification Model 2 - Random Forest

In [67]:
# Add parameters of your choice here:
RF_classifier = RandomForestClassifier()
RF_paramGrid = (ParamGridBuilder() \
               .addGrid(RF_classifier.maxDepth, [2, 5, 10]) \
#                .addGrid(RF_classifier.maxBins, [5, 10, 20]) \
                .addGrid(RF_classifier.numTrees, [5, 20, 40]) \
                .build())

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Exception in thread cell_monitor-67:
Traceback (most recent call last):
  File "/mnt/notebook-env/lib/python3.9/threading.py", line 980, in _bootstrap_inner
    self.run()
  File "/mnt/notebook-env/lib/python3.9/threading.py", line 917, in run
    self._target(*self._args, **self._kwargs)
  File "/mnt/notebook-env/lib/python3.9/site-packages/awseditorssparkmonitoringwidget/cellmonitor.py", line 154, in cell_monitor
    job_group_filtered_jobs = [job for job in jobs_data if job['jobGroup'] == str(statement_id)]
  File "/mnt/notebook-env/lib/python3.9/site-packages/awseditorssparkmonitoringwidget/cellmonitor.py", line 154, in <listcomp>
    job_group_filtered_jobs = [job for job in jobs_data if job['jobGroup'] == str(statement_id)]
KeyError: 'jobGroup'


In [68]:
#Cross Validator requires all of the following parameters:
RF_crossval = CrossValidator(estimator=RF_classifier,
                          estimatorParamMaps=RF_paramGrid,
                          evaluator=Bin_evaluator,
                          numFolds=5) # 3 + is best practice

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Exception in thread cell_monitor-68:
Traceback (most recent call last):
  File "/mnt/notebook-env/lib/python3.9/threading.py", line 980, in _bootstrap_inner
    self.run()
  File "/mnt/notebook-env/lib/python3.9/threading.py", line 917, in run
    self._target(*self._args, **self._kwargs)
  File "/mnt/notebook-env/lib/python3.9/site-packages/awseditorssparkmonitoringwidget/cellmonitor.py", line 154, in cell_monitor
    job_group_filtered_jobs = [job for job in jobs_data if job['jobGroup'] == str(statement_id)]
  File "/mnt/notebook-env/lib/python3.9/site-packages/awseditorssparkmonitoringwidget/cellmonitor.py", line 154, in <listcomp>
    job_group_filtered_jobs = [job for job in jobs_data if job['jobGroup'] == str(statement_id)]
KeyError: 'jobGroup'


In [69]:
# Fit Model: Run cross-validation, and choose the best set of parameters.
fit_RF_Model = RF_crossval.fit(train_df)

# Retrieve best model from cross val
Best_RF_Model = fit_RF_Model.bestModel
featureImportances = Best_RF_Model.featureImportances.toArray()
print("Feature Importances: ",featureImportances)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Exception in thread cell_monitor-69:
Traceback (most recent call last):
  File "/mnt/notebook-env/lib/python3.9/threading.py", line 980, in _bootstrap_inner
    self.run()
  File "/mnt/notebook-env/lib/python3.9/threading.py", line 917, in run
    self._target(*self._args, **self._kwargs)
  File "/mnt/notebook-env/lib/python3.9/site-packages/awseditorssparkmonitoringwidget/cellmonitor.py", line 154, in cell_monitor
    job_group_filtered_jobs = [job for job in jobs_data if job['jobGroup'] == str(statement_id)]
  File "/mnt/notebook-env/lib/python3.9/site-packages/awseditorssparkmonitoringwidget/cellmonitor.py", line 154, in <listcomp>
    job_group_filtered_jobs = [job for job in jobs_data if job['jobGroup'] == str(statement_id)]
KeyError: 'jobGroup'


Feature Importances:  [5.93024305e-02 1.16145260e-01 1.20896468e-01 0.00000000e+00
 2.91892989e-02 2.63608896e-01 2.55172618e-01 1.93513290e-05
 9.02309273e-02 4.65428999e-05 3.87312019e-04 6.50008946e-02]

In [70]:
# RF Model predictions on test data
RF_predictions = fit_RF_Model.transform(test_df)

# Check accuracy
RF_accuracy = (Bin_evaluator.evaluate(RF_predictions))*100
print(" ")
print("Accuracy: ", RF_accuracy)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Exception in thread cell_monitor-70:
Traceback (most recent call last):
  File "/mnt/notebook-env/lib/python3.9/threading.py", line 980, in _bootstrap_inner
    self.run()
  File "/mnt/notebook-env/lib/python3.9/threading.py", line 917, in run
    self._target(*self._args, **self._kwargs)
  File "/mnt/notebook-env/lib/python3.9/site-packages/awseditorssparkmonitoringwidget/cellmonitor.py", line 154, in cell_monitor
    job_group_filtered_jobs = [job for job in jobs_data if job['jobGroup'] == str(statement_id)]
  File "/mnt/notebook-env/lib/python3.9/site-packages/awseditorssparkmonitoringwidget/cellmonitor.py", line 154, in <listcomp>
    job_group_filtered_jobs = [job for job in jobs_data if job['jobGroup'] == str(statement_id)]
KeyError: 'jobGroup'


 
Accuracy:  84.76191862123893

In [75]:
#Generate Confusion Matrix to review results of predictions
#Cast to float type, and order by prediction, else it won't work
RF_preds_and_labels = RF_predictions.select(['prediction','label']).withColumn('label', F.col('label').cast(FloatType())).orderBy('prediction')

#select only prediction and label columns
RF_preds_and_labels = RF_preds_and_labels.select(['prediction','label'])

from pyspark.mllib.evaluation import MulticlassMetrics
RF_metrics = MulticlassMetrics(RF_preds_and_labels.rdd.map(tuple))

print(RF_metrics.confusionMatrix().toArray())

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Exception in thread cell_monitor-75:
Traceback (most recent call last):
  File "/mnt/notebook-env/lib/python3.9/threading.py", line 980, in _bootstrap_inner
    self.run()
  File "/mnt/notebook-env/lib/python3.9/threading.py", line 917, in run
    self._target(*self._args, **self._kwargs)
  File "/mnt/notebook-env/lib/python3.9/site-packages/awseditorssparkmonitoringwidget/cellmonitor.py", line 154, in cell_monitor
    job_group_filtered_jobs = [job for job in jobs_data if job['jobGroup'] == str(statement_id)]
  File "/mnt/notebook-env/lib/python3.9/site-packages/awseditorssparkmonitoringwidget/cellmonitor.py", line 154, in <listcomp>
    job_group_filtered_jobs = [job for job in jobs_data if job['jobGroup'] == str(statement_id)]
KeyError: 'jobGroup'


[[1.534496e+06 3.700000e+01]
 [7.140000e+02 1.629000e+03]]
/mnt/yarn/usercache/livy/appcache/application_1731984268094_0001/container_1731984268094_0001_01_000001/pyspark.zip/pyspark/sql/context.py:158: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.

In [77]:
print("Summary of evaluation metrics for Random Forest model:")
print("Accuracy: ", RF_metrics.accuracy)
print("Precision: ", RF_metrics.weightedPrecision)
print("Recall: ", RF_metrics.weightedRecall)
print("F1 Score: ", RF_metrics.weightedFMeasure())

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Summary of evaluation metrics for Random Forest model:
Accuracy:  0.9995113463935933
Precision:  0.9995017681628143
Recall:  0.9995113463935932
F1 Score:  0.9994701407997197

Exception in thread cell_monitor-77:
Traceback (most recent call last):
  File "/mnt/notebook-env/lib/python3.9/threading.py", line 980, in _bootstrap_inner
    self.run()
  File "/mnt/notebook-env/lib/python3.9/threading.py", line 917, in run
    self._target(*self._args, **self._kwargs)
  File "/mnt/notebook-env/lib/python3.9/site-packages/awseditorssparkmonitoringwidget/cellmonitor.py", line 154, in cell_monitor
    job_group_filtered_jobs = [job for job in jobs_data if job['jobGroup'] == str(statement_id)]
  File "/mnt/notebook-env/lib/python3.9/site-packages/awseditorssparkmonitoringwidget/cellmonitor.py", line 154, in <listcomp>
    job_group_filtered_jobs = [job for job in jobs_data if job['jobGroup'] == str(statement_id)]
KeyError: 'jobGroup'


The Random Forest model had weighted metrics (Accuracy, Precision, Recall, F1) of 99.9% across the board.

And the initial accuracy of 84.76% tells us that it is performing well and not overfitting to the non-fraud rows.

-----------------------

#### Classification Model 3 - Decision Tree

In [79]:
# Add parameters of your choice here:
DT_classifier = DecisionTreeClassifier()
DT_paramGrid = (ParamGridBuilder() \
             .addGrid(DT_classifier.maxDepth, [2, 5, 10, 15, 20]) \
             .addGrid(DT_classifier.maxBins, [10, 20, 40, 60, 80]) \
             .build())

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Exception in thread cell_monitor-79:
Traceback (most recent call last):
  File "/mnt/notebook-env/lib/python3.9/threading.py", line 980, in _bootstrap_inner
    self.run()
  File "/mnt/notebook-env/lib/python3.9/threading.py", line 917, in run
    self._target(*self._args, **self._kwargs)
  File "/mnt/notebook-env/lib/python3.9/site-packages/awseditorssparkmonitoringwidget/cellmonitor.py", line 154, in cell_monitor
    job_group_filtered_jobs = [job for job in jobs_data if job['jobGroup'] == str(statement_id)]
  File "/mnt/notebook-env/lib/python3.9/site-packages/awseditorssparkmonitoringwidget/cellmonitor.py", line 154, in <listcomp>
    job_group_filtered_jobs = [job for job in jobs_data if job['jobGroup'] == str(statement_id)]
KeyError: 'jobGroup'


In [80]:
#Cross Validator requires all of the following parameters:
DT_crossval = CrossValidator(estimator=DT_classifier,
                          estimatorParamMaps=DT_paramGrid,
                          evaluator=Bin_evaluator,
                          numFolds=5) # 3 + is best practice

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Exception in thread cell_monitor-80:
Traceback (most recent call last):
  File "/mnt/notebook-env/lib/python3.9/threading.py", line 980, in _bootstrap_inner
    self.run()
  File "/mnt/notebook-env/lib/python3.9/threading.py", line 917, in run
    self._target(*self._args, **self._kwargs)
  File "/mnt/notebook-env/lib/python3.9/site-packages/awseditorssparkmonitoringwidget/cellmonitor.py", line 154, in cell_monitor
    job_group_filtered_jobs = [job for job in jobs_data if job['jobGroup'] == str(statement_id)]
  File "/mnt/notebook-env/lib/python3.9/site-packages/awseditorssparkmonitoringwidget/cellmonitor.py", line 154, in <listcomp>
    job_group_filtered_jobs = [job for job in jobs_data if job['jobGroup'] == str(statement_id)]
KeyError: 'jobGroup'


In [ ]:
# Fit Model: Run cross-validation, and choose the best set of parameters.
fit_DT_Model = DT_crossval.fit(train_df)

# Collect and print feature importances
Best_DT_Model = fit_DT_Model.bestModel
DT_featureImportances = Best_DT_Model.featureImportances.toArray()
print("Feature Importances: ",DT_featureImportances)

DT_predictions = fit_DT_Model.transform(test_df)
DT_accuracy = (Bin_evaluator.evaluate(DT_predictions))*100
print("Accuracy: ",DT_accuracy)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Exception in thread cell_monitor-81:
Traceback (most recent call last):
  File "/mnt/notebook-env/lib/python3.9/threading.py", line 980, in _bootstrap_inner
    self.run()
  File "/mnt/notebook-env/lib/python3.9/threading.py", line 917, in run
    self._target(*self._args, **self._kwargs)
  File "/mnt/notebook-env/lib/python3.9/site-packages/awseditorssparkmonitoringwidget/cellmonitor.py", line 154, in cell_monitor
    job_group_filtered_jobs = [job for job in jobs_data if job['jobGroup'] == str(statement_id)]
  File "/mnt/notebook-env/lib/python3.9/site-packages/awseditorssparkmonitoringwidget/cellmonitor.py", line 154, in <listcomp>
    job_group_filtered_jobs = [job for job in jobs_data if job['jobGroup'] == str(statement_id)]
KeyError: 'jobGroup'


Feature Importances:  [0.05896556 0.06371143 0.36019011 0.         0.04487201 0.42130889
 0.01500934 0.         0.01239663 0.         0.         0.02354603]
Accuracy:  87.91872033977117

In [82]:
#Generate Confusion Matrix to review results of predictions
#Cast to float type, and order by prediction, else it won't work
DT_preds_and_labels = DT_predictions.select(['prediction','label']).withColumn('label', F.col('label').cast(FloatType())).orderBy('prediction')

#select only prediction and label columns
DT_preds_and_labels = DT_preds_and_labels.select(['prediction','label'])

from pyspark.mllib.evaluation import MulticlassMetrics
DT_metrics = MulticlassMetrics(DT_preds_and_labels.rdd.map(tuple))

print(DT_metrics.confusionMatrix().toArray())

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Exception in thread cell_monitor-82:
Traceback (most recent call last):
  File "/mnt/notebook-env/lib/python3.9/threading.py", line 980, in _bootstrap_inner
    self.run()
  File "/mnt/notebook-env/lib/python3.9/threading.py", line 917, in run
    self._target(*self._args, **self._kwargs)
  File "/mnt/notebook-env/lib/python3.9/site-packages/awseditorssparkmonitoringwidget/cellmonitor.py", line 154, in cell_monitor
    job_group_filtered_jobs = [job for job in jobs_data if job['jobGroup'] == str(statement_id)]
  File "/mnt/notebook-env/lib/python3.9/site-packages/awseditorssparkmonitoringwidget/cellmonitor.py", line 154, in <listcomp>
    job_group_filtered_jobs = [job for job in jobs_data if job['jobGroup'] == str(statement_id)]
KeyError: 'jobGroup'


[[1.534454e+06 2.240000e+02]
 [5.810000e+02 1.825000e+03]]

In [83]:
print("Summary of evaluation metrics for Decision Tree model:")
print("Accuracy: ", DT_metrics.accuracy)
print("Precision: ", DT_metrics.weightedPrecision)
print("Recall: ", DT_metrics.weightedRecall)
print("F1 Score: ", DT_metrics.weightedFMeasure())

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Summary of evaluation metrics for Decision Tree model:
Accuracy:  0.999476281062063
Precision:  0.9994509781531474
Recall:  0.999476281062063
F1 Score:  0.9994553274908828

Exception in thread cell_monitor-83:
Traceback (most recent call last):
  File "/mnt/notebook-env/lib/python3.9/threading.py", line 980, in _bootstrap_inner
    self.run()
  File "/mnt/notebook-env/lib/python3.9/threading.py", line 917, in run
    self._target(*self._args, **self._kwargs)
  File "/mnt/notebook-env/lib/python3.9/site-packages/awseditorssparkmonitoringwidget/cellmonitor.py", line 154, in cell_monitor
    job_group_filtered_jobs = [job for job in jobs_data if job['jobGroup'] == str(statement_id)]
  File "/mnt/notebook-env/lib/python3.9/site-packages/awseditorssparkmonitoringwidget/cellmonitor.py", line 154, in <listcomp>
    job_group_filtered_jobs = [job for job in jobs_data if job['jobGroup'] == str(statement_id)]
KeyError: 'jobGroup'


It is very close, but I think the Decision Tree model may have performed the best of the three classification models.

Like Random Forest, its scores for the weighted evaluation metrics (Accuracy, Precision, Recall, F1) were 99.9% across the board.

The differentiating factor was the initial accuracy that we calculated, which was 87.92%.